# Phase 3: Fuzzy Systems — `FS_Model.ipynb`**Team 1 — NYC Taxi Trip Duration**

---## Words We Use in This Notebook (Glossary)Before reading the code, here are the main terms used in this phase.| Term | Meaning ||------|---------|| **Fuzzy Set** | A set where membership is not binary (0 or 1) but a degree between 0 and 1. For example, a 12 km trip might be 0.7 "medium" and 0.3 "long." || **Membership Function (MF)** | A curve that maps a crisp input value to a membership degree (0–1). Common shapes: triangular, trapezoidal, Gaussian. || **Linguistic Variable** | A variable described with words instead of numbers — e.g., trip distance is "short," "medium," or "long." || **Fuzzy Rule** | An IF–THEN statement using linguistic variables — e.g., IF distance IS long AND hour IS rush_hour THEN duration IS very_long. || **Fuzzy Inference System (FIS)** | The complete system: fuzzify inputs → apply rules → defuzzify to get a crisp predicted value. || **Defuzzification** | Converting fuzzy output back into a single number (e.g., centroid method). || **scikit-fuzzy** | Python library (`skfuzzy`) for building fuzzy inference systems. || **Interpretability** | How easy it is for a human to understand *why* the model made a prediction. Measured by rule count, rule length, and feature count. || **H3** | Phase 3 hypothesis: Does the fuzzy system provide meaningful interpretability while maintaining acceptable predictive performance compared to the NN (Phase 1) and EA (Phase 2)? |

---### What this notebook doesIn this notebook, we build a **Fuzzy Inference System (FIS)** to predict NYC taxi trip duration.Unlike the neural network (Phase 1) and the evolutionary algorithm (Phase 2), the fuzzy system is designed for **interpretability** — the goal is a model whose reasoning a human can read and understand, expressed as simple IF–THEN rules using everyday language like "short distance" or "rush hour."We intentionally trade some prediction accuracy for transparency. The fuzzy system will not beat the NN on raw metrics, and that is expected and acceptable. The value is in producing a model where every prediction can be explained.**Phases at a glance:**| Phase | Method | Notebook | Primary Goal ||-------|--------|----------|-------------|| 1 | Neural Network | `NN_Analysis.ipynb` | Maximize predictive accuracy || 2 | Evolutionary Algorithm | `EA_Optimization.ipynb` | Optimize NN hyperparameters via GA || 3 | **Fuzzy System** | **`FS_Model.ipynb` (this notebook)** | **Interpretable prediction via fuzzy rules** |

---## How to Run This Notebook1. **Open the notebook in NRP JupyterHub** (required for final results).   You may test locally, but all final outputs must be executed on NRP with the PyTorch2 environment.2. **Run cells in order from top to bottom.** Each cell depends on earlier cells.3. **Do not modify constants** (seed, split sizes, data path) — these must match Phases 1 and 2.### Execution environmentThe fuzzy system is implemented in **Python using scikit-fuzzy (`skfuzzy`)**, not TensorFlow or PyTorch. However, the notebook runs inside the NRP PyTorch2 environment as required by the project spec. The stack requirement is about the execution environment, not the modeling library.

---## Fair Evaluation RulesTo ensure honest and comparable results across all three phases, we follow strict data usage rules:- **Same dataset:** First 1,000,000 rows from `train.csv`- **Same split:** 70% train / 15% validation / 15% test (identical to Phases 1 and 2)- **Same seed:** All random operations use the same fixed seed as prior phases- **No re-shuffling:** Data order is identical to Phases 1 and 2- **Test set untouched:** The 15% test set is used **exactly once** for final evaluation in Step 3- **Validation only for design/tuning:** All membership function tuning and rule refinement uses the validation set- **No target leakage:** No feature depends on future or target information

---# Step 1: Fuzzy System Design & Setup## 1.1 — Fuzzy System Objective**Prediction target:** `trip_duration` (seconds), consistent with Phases 1 and 2.**Purpose:** Build a fuzzy inference system that provides **human-readable, rule-based predictions** for NYC taxi trip duration. The system intentionally trades some predictive accuracy for **interpretability** — every prediction can be traced back to a small set of IF–THEN rules using linguistically meaningful terms (e.g., "IF distance IS long AND pickup_hour IS rush_hour THEN duration IS very_long").**Hypothesis (H3):** The fuzzy system provides meaningful interpretability (measured by number of rules, average rule length, and feature coverage) while maintaining acceptable predictive performance compared to the Phase 1 NN and Phase 2 EA-optimized NN.**Evaluation path:**- **During development (Steps 1–2):** All design decisions, membership function tuning, and rule validation use the **validation set only**.- **Final evaluation (Step 3):** The fuzzy system is evaluated **once** on the held-out 15% test set, and metrics (RMSE, MAE, R², MAPE) are compared against Phase 1 and Phase 2 results.**What "acceptable" means:** We do not require the fuzzy system to match or exceed the NN's R². A meaningful drop in accuracy is justified if the model is fully transparent — i.e., a non-expert can read the rules and understand why a given trip was predicted to be long or short.

---## 1.2 — Feature Selection: Evidence from Phases 1 and 2### Why we select features carefullyFuzzy systems scale poorly with many inputs. Each input feature multiplies the potential rule space. The project FAQ explicitly warns about this. We target **6 input features** to keep the rule base manageable and interpretable.### Phase 1 NN Permutation Importance (Validation Set, R²_log drop)These are the features ranked by how much validation R² drops when the feature is randomly shuffled — higher drop means more important:| Rank | Feature | R²_log Drop | Std | Family ||------|---------|------------|-----|--------|| 1 | `haversine_km` | **2.0482** | 0.0064 | Spatial || 2 | `hour_cos` | 0.3014 | 0.0006 | Temporal || 3 | `delta_lat` | 0.2860 | 0.0011 | Spatial || 4 | `pickup_hour` | 0.1077 | 0.0007 | Temporal || 5 | `dow_cos` | 0.0846 | 0.0004 | Temporal || 6 | `delta_lon` | 0.0644 | 0.0010 | Spatial || 7 | `pickup_dow` | 0.0477 | 0.0003 | Temporal || 8 | `passenger_count` | 0.0038 | 0.0002 | Context || ... | `vendor_1/2`, `store_and_fwd`, `pickup_month` | < 0.004 | — | Various |### Grouped importance (NN)| Family | Total R²_log Drop | Interpretation ||--------|-------------------|---------------|| Spatial | **2.399** | Dominant — distance and direction are by far the strongest predictors || Temporal | **0.639** | Strong — time of day and day of week matter significantly || Context | **0.123** | Weak — vendor and passenger info contribute minimally |### Phase 2 EA findingsThe EA-optimized NN did **not** improve over the Phase 1 NN (ΔR² = −0.026, bootstrap CI includes 0). This confirms the Phase 1 feature set and architecture were already near-optimal, and reinforces that Phase 3 should focus on interpretability rather than accuracy gains.

---## 1.3 — Selected Input Features (6 features)Based on the Phase 1/2 evidence above, we select the following 6 features as fuzzy input variables:| # | Feature | Family | NN Importance Rank | R²_log Drop | Why Selected | Expected Influence on Duration ||---|---------|--------|-------------------|------------|-------------|-------------------------------|| 1 | `haversine_km` | Spatial | 1 | 2.048 | By far the most important feature across both models. Directly interpretable as "how far is the trip." | Longer distance → longer trip duration || 2 | `pickup_hour` | Temporal | 4 | 0.108 | Raw hour (0–23) maps naturally to fuzzy labels like "early morning," "morning commute," "midday," "evening rush," "night." More interpretable than sin/cos encoding. | Rush hours → longer trips due to traffic; late night → shorter trips || 3 | `pickup_dow` | Temporal | 7 | 0.048 | Day of week (0=Mon, 6=Sun) maps to "weekday" vs "weekend" patterns. More intuitive than sin/cos for fuzzy rules. | Weekdays (esp. Mon–Fri) → more traffic → longer trips || 4 | `delta_lat` | Spatial | 3 | 0.286 | North–south displacement. Captures directional travel patterns the NN found important (rank 3). | Large absolute displacement → longer trip || 5 | `delta_lon` | Spatial | 6 | 0.064 | East–west displacement. Together with delta_lat, captures travel direction without the complexity of haversine alone. | Large absolute displacement → longer trip || 6 | `passenger_count` | Context | 8 | 0.004 | Weakest selected feature, but the only interpretable context variable. Included to represent non-spatial, non-temporal factors. | Minimal direct effect; may correlate with trip type |### Features NOT selected (and why)| Feature | Why Excluded ||---------|-------------|| `hour_sin`, `hour_cos` | Cyclic encodings are mathematically useful for NNs but not linguistically interpretable. We use raw `pickup_hour` instead, which maps naturally to fuzzy labels. || `dow_sin`, `dow_cos` | Same reasoning — raw `pickup_dow` is more natural for fuzzy rules ("weekday" vs "weekend"). || `pickup_month` | Low importance in both models (NN rank 12, Ridge rank 5). Adds complexity without proportional interpretability gain. || `vendor_1`, `vendor_2` | Binary categorical. Not meaningful as fuzzy variables — vendor identity has no ordinal or continuous structure. || `store_and_fwd_Y` | Negligible importance (NN drop = 0.0001). Nearly zero variance. |

---## 1.4 — Output Variable Design**Output variable:** `predicted_duration` (seconds)The output uses **5 fuzzy labels** spanning typical NYC taxi trip durations. Boundaries are informed by the observed distribution in the training data (Phase 1), where most trips fall between 3–30 minutes with a right-skewed tail.| Label | Approximate Range (seconds) | Approximate Range (minutes) | Description ||-------|---------------------------|---------------------------|-------------|| `very_short` | 0 – 300 | 0 – 5 min | Quick trips: same-neighborhood, short hops || `short` | 120 – 900 | 2 – 15 min | Typical short urban trips || `medium` | 600 – 1800 | 10 – 30 min | Average-length trips, moderate distance || `long` | 1200 – 3600 | 20 – 60 min | Cross-borough or congested trips || `very_long` | 2400 – 5400+ | 40 – 90+ min | Airport runs, outer-borough, or heavy traffic |**Notes:**- Ranges deliberately **overlap** — this is fundamental to fuzzy logic. A 15-minute trip might be 0.4 "short" and 0.6 "medium" simultaneously.- Membership functions will be **triangular** by default (simplest, most interpretable). Brandon will refine shapes and exact boundaries from the actual training data distribution in Step 1.- **Defuzzification method:** Centroid (center of gravity) — the most common and interpretable approach.- The exact numeric boundaries will be finalized after Brandon completes the `membership_range_table` using observed data percentiles.

---## 1.5 — Input Variable Membership Function Design (Initial Proposal)For each input feature, we propose initial linguistic labels and membership function shapes. These are starting points — Brandon will refine the exact boundaries using observed data distributions.### `haversine_km` — Trip Distance| Label | Approx Range (km) | Shape ||-------|-------------------|-------|| `short` | 0 – 3 | Triangular || `medium` | 1.5 – 8 | Triangular || `long` | 5 – 20+ | Triangular |*Rationale:* 3 labels. Most NYC taxi trips are under 10 km. Distance is the dominant predictor (R²_log drop = 2.048).### `pickup_hour` — Hour of Day (0–23)| Label | Approx Range | Shape ||-------|-------------|-------|| `early_morning` | 0 – 7 | Triangular || `morning_rush` | 6 – 10 | Triangular || `midday` | 9 – 16 | Triangular || `evening_rush` | 15 – 20 | Triangular || `night` | 19 – 24 | Triangular |*Rationale:* 5 labels to capture NYC traffic patterns. Rush hours have distinct duration profiles. This is the strongest temporal predictor.### `pickup_dow` — Day of Week (0=Mon, 6=Sun)| Label | Approx Range | Shape ||-------|-------------|-------|| `weekday` | 0 – 4 | Trapezoidal || `weekend` | 4.5 – 6 | Trapezoidal |*Rationale:* 2 labels. Weekday vs weekend is the most natural and interpretable split. Trapezoidal to give full membership across Mon–Thu and Sat–Sun, with Friday as a transition.### `delta_lat` — North–South Displacement| Label | Approx Range | Shape ||-------|-------------|-------|| `south` | −0.15 – −0.01 | Triangular || `neutral` | −0.03 – 0.03 | Triangular || `north` | 0.01 – 0.15 | Triangular |*Rationale:* 3 labels. Captures directional travel patterns in Manhattan's grid.### `delta_lon` — East–West Displacement| Label | Approx Range | Shape ||-------|-------------|-------|| `west` | −0.15 – −0.01 | Triangular || `neutral` | −0.03 – 0.03 | Triangular || `east` | 0.01 – 0.15 | Triangular |*Rationale:* 3 labels. Same logic as delta_lat for the east–west axis.### `passenger_count` — Number of Passengers| Label | Approx Range | Shape ||-------|-------------|-------|| `solo` | 0 – 2 | Triangular || `small_group` | 1.5 – 4 | Triangular || `large_group` | 3.5 – 6+ | Triangular |*Rationale:* 3 labels. Most trips are 1–2 passengers. Included for context diversity, not predictive power.

---## 1.6 — Interpretability PlanInterpretability is the primary justification for Phase 3. We define specific, measurable targets aligned with the project FAQ.### Structural targets| Metric | Target | Rationale ||--------|--------|-----------|| **Number of input features** | 6 | FAQ warns fuzzy systems scale poorly; 6 is within the recommended 5–8 range || **Max number of rules** | 30–50 | With 6 inputs and mostly 3 labels each, the full combinatorial space is ~3×5×2×3×3×3 = 810 rules. We will prune aggressively to keep only meaningful rules. || **Average rule length** | 2–3 antecedents | Rules with too many conditions are hard to interpret. Most rules should use 2–3 input variables. || **Feature coverage** | 100% | Every selected input should appear in at least one active rule. || **Rule redundancy** | Minimal | No two rules should produce effectively identical outputs for the same region of input space. |### How interpretability will be reported (Step 3)- Total active rule count- Average and max rule length (number of antecedents)- Coverage: fraction of test samples that activate at least one rule- Example rules in plain English (e.g., "IF distance IS long AND hour IS evening_rush THEN duration IS very_long")- Comparison: number of interpretable parameters (rules × antecedents) vs NN parameter count

---## 1.7 — Methodology LockThe following constraints are locked for Phase 3 and must not be changed during implementation:| Constraint | Value | Reason ||-----------|-------|--------|| **Dataset** | First 1,000,000 rows of `train.csv` | Same as Phases 1 and 2 || **Split** | 70/15/15 train/val/test with same seed | Reuse exact Phase 1 split — no re-shuffling || **Test set** | Untouched until Step 3 final evaluation | Prevents overfitting to test data || **Validation use** | Design, tuning, rule refinement only | All Step 1–2 decisions validated here || **Implementation** | Python + scikit-fuzzy (`skfuzzy`) | NOT TensorFlow-only. The fuzzy system must be a real fuzzy inference system, not a neural surrogate. || **Execution environment** | NRP JupyterHub (PyTorch2 stack) | Stack requirement is about the environment, not the modeling library || **Random seed** | Same as Phases 1 and 2 | Reproducibility || **Defuzzification** | Centroid method | Most common, most interpretable |### Compliance statementThis Phase 3 implementation reuses the controlled data split from Phases 1 and 2 without re-shuffling. The test set remains untouched until the one-time final evaluation in Step 3. The fuzzy system is implemented as a genuine fuzzy inference system using scikit-fuzzy, running within the NRP PyTorch2 environment. No features introduce target leakage or depend on future information.

---## Step 1 Complete — Handoff Summary### Deliverables completed| Deliverable | Status | Owner ||------------|--------|-------|| FS objective statement | ✅ Complete (Section 1.1) | Nadir || Feature selection with evidence | ✅ Complete (Sections 1.2–1.3) | Nadir || Output variable design | ✅ Complete (Section 1.4) | Nadir || Input membership function proposal | ✅ Complete (Section 1.5) | Nadir || Interpretability plan | ✅ Complete (Section 1.6) | Nadir || Methodology lock | ✅ Complete (Section 1.7) | Nadir |### What happens next (Step 2 — Implementation, due Apr 16)- **Brandon:** Refine membership boundaries using actual data distributions; finalize `membership_range_table`- **Dominic:** Polish design tables into final notebook/wiki documentation; QA review- **Iran:** Audit data compliance; verify no leakage; write Phase 3 protocol block- **David:** Approve final feature list; signoff on Step 1- **All:** Begin scikit-fuzzy implementation and rule base construction### Pending team handoffs- [ ] David approves feature list (Section 1.3)- [ ] Brandon delivers refined membership ranges from observed data- [ ] Dominic QA review of design clarity- [ ] Iran compliance audit signoff

------# Step 2: Fuzzy System Implementation*(Step 2 code cells begin below — due Thursday, April 16)*> **Status:** Awaiting Step 1 approval and Brandon's refined membership ranges before implementation begins.

---## Cell 1 — Imports, Constants & Environment> **What this will do:** Load all required libraries (numpy, pandas, skfuzzy, sklearn) and define constants matching Phases 1 and 2.

In [ ]:
# Step 2 — To be implemented after Step 1 approval# Imports and constants will go here# Must match Phase 1/2: SEED, NROWS, DATA_PATH, split ratios

---## Cell 2 — Data Load & Split Verification> **What this will do:** Load the same 1M rows, apply the same split, verify row counts match Phases 1 and 2.

In [ ]:
# Step 2 — To be implemented# Reuse exact data loading and splitting from Phase 1# Verify: train ~700k, val ~150k, test ~150k

---## Cell 3 — Feature Engineering (Same Pipeline as Phases 1 & 2)> **What this will do:** Apply the same `build_features()` function from Phase 1, then select only the 6 fuzzy input features.

In [ ]:
# Step 2 — To be implemented# Apply build_features() from Phase 1# Select only: haversine_km, pickup_hour, pickup_dow, delta_lat, delta_lon, passenger_count

---## Cell 4 — Define Membership Functions> **What this will do:** Define fuzzy universes and membership functions for all 6 inputs and the output using skfuzzy.

In [ ]:
# Step 2 — To be implemented# import skfuzzy as fuzz# import skfuzzy.control as ctrl# Define Antecedent/Consequent variables with membership functions

---## Cell 5 — Define Fuzzy Rule Base> **What this will do:** Create the IF–THEN fuzzy rules based on domain knowledge and data patterns.

In [ ]:
# Step 2 — To be implemented# Define fuzzy rules using ctrl.Rule()# Target: 30-50 meaningful rules with 2-3 antecedents each

---## Cell 6 — Build & Run Fuzzy Inference System> **What this will do:** Create the control system, run predictions on validation set, compute metrics.

In [ ]:
# Step 2 — To be implemented# Build ControlSystem and ControlSystemSimulation# Predict on validation set# Compute RMSE, MAE, R², MAPE on validation